# ENGR 1451/2451 — Weekly Assignment 7
**10 points · 4 graded parts**  
**Due:** See Canvas.

Practice confounding, standard error, and confidence intervals. Show your Python code and keep written answers brief.


## What to hand in

Work with the **course tutor** in the class project. Use it as your AI help; it keeps the record, so no separate log is needed. Your code and written answers must still be your own.

At the end of each working session, type **"make my record"** in the chat. Copy its block into **Session records** below, placing each new record after the previous one.

To continue later, start a new chat in the same project and state the assignment and where you stopped. Session records are not graded for correctness or how much help you needed.


## Setup

All data are embedded; no additional files are needed. Keep full precision in calculations and round only displayed results.


In [4]:
import numpy as np
import pandas as pd
import scipy.stats as stats

pd.set_option("display.precision", 4)


## LE7-1. Group composition and confounding (4 pts)

These synthetic inspection records compare two manufacturing methods. Methods were **not randomly assigned** to parts.

Calculate pass rates for each method within each part type and overall. For overall rates, use `groupby()` and sum the counts before dividing; do not average percentages.

Which method has the higher overall rate? Which has the higher rate within each part type? In two sentences, explain the reversal using group composition and why these observations do not establish a causal effect of method.

**Answer:**


In [5]:
inspection = pd.DataFrame({
    "method": ["A", "A", "B", "B"],
    "part_type": ["Simple", "Complex", "Simple", "Complex"],
    "inspected": [20, 80, 80, 20],
    "passed": [18, 48, 64, 10],
})
# Calculate and display pass rates within each part type.
inspection["pass_rate"] = inspection["passed"]/inspection["inspected"]
display(inspection[["method","part_type","pass_rate"]])

# Group by method, sum inspected/passed counts, and calculate overall rates.
#still has the pass rates column which is now wrong because the rates should not be summed
grouped_inspection = inspection.groupby("method").sum()

#overwrite the pass_rate column with pass rates by method group
grouped_inspection["pass_rate"] = grouped_inspection["passed"]/grouped_inspection["inspected"] #express as %
display(grouped_inspection["pass_rate"])


,method,part_type,pass_rate
0,A,Simple,0.9
1,A,Complex,0.6
2,B,Simple,0.8
3,B,Complex,0.5


method
A    0.66
B    0.74
Name: pass_rate, dtype: float64

When split up into simple and complex parts in the original data, you can see that the simple parts have a higher pass rate than complex parts for both methods. Also, the pass rate by type is higher for each in method A than method B. When part types are combined, however, the trend reverses and it looks like method B has a higher overall pass rate. The fact that the method b group contained more simple parts that tended to pass more contributed to this. One possible reason could be the fact that groups were not randomly assigned. This means that a technician, for example, would choose a method for a part with the insight that the method may work better for the particular design of the part.

## LE7-2. Estimate a population mean (4 pts)

The synthetic tensile strengths below are in **MPa**. Assume an independent random sample from a stable, approximately normal production process. Its population mean and SD are unknown.

### A. Estimate the SD and SE (2 pts)

Display $n$, $\bar x$, sample SD $s$ using `std(ddof=1)`, and estimated SE using `stats.sem(..., ddof=1)`.

In one sentence, distinguish what $s$ and $s/\sqrt{n}$ measure.

**Answer:**


In [6]:
strength = np.array([
    48.2, 51.4, 49.7, 52.1, 46.8, 50.5, 53.0, 47.6,
    49.1, 51.8, 48.9, 50.7, 47.9, 52.5, 49.8, 50.2,
])

# Calculate and display n, sample_mean, sample_sd, and estimated_se.
strength_stats = pd.Series([
    strength.size,
    strength.mean(),
    strength.std(ddof=1),
    stats.sem(strength, ddof = 1)
],
    index = ["Count","Mean","Sample s","Standard Error"]
)
display(strength_stats)
n = strength_stats["Count"]

Count             16.0000
Mean              50.0125
Sample s           1.8518
Standard Error     0.4630
dtype: float64

Standard deviation, s, measures the spread of one sample around its mean. Meanwhile the standard error, SE, is the standard deviation of the sample mean across multiple samples.

### B. Construct and interpret a confidence interval (2 pts)

Because $\sigma$ is unknown, use a **Student's $t$ interval**:

$$\bar x \pm t^*\frac{s}{\sqrt{n}},\qquad df=n-1.$$

Calculate `t_star = stats.t.ppf(0.975, df=n-1)` for 95% confidence. Display $df$, $t^*$, and both interval endpoints in MPa.

Interpret the interval for the **population mean**. Does it describe where 95% of individual strengths fall? Would a 99% interval using the same sample be wider or narrower? No second calculation is required.

**Answer:**


In [7]:
# Calculate the t critical value and margin of error.
alpha = 1-0.95
t_star = stats.t.ppf(1-alpha/2, df = n-1)
print(t_star)
lower_endpt = strength_stats["Mean"] - t_star*(strength_stats["Sample s"]/np.sqrt(n))
upper_endpt = strength_stats["Mean"] + t_star*(strength_stats["Sample s"]/np.sqrt(n))
ME = t_star*strength_stats["Standard Error"]
# Display df, t_star, and the lower/upper confidence limits in MPa.
confidence_stats = pd.Series([
    (n-1),
    t_star,
    lower_endpt,
    upper_endpt
],
    index = ["degrees of freedom","t *","lower confidence limit (MPa)","upper confidence limit (MPa)"]
)
display(confidence_stats)

2.131449545559323


degrees of freedom              15.0000
t *                              2.1314
lower confidence limit (MPa)    49.0257
upper confidence limit (MPa)    50.9993
dtype: float64

The interval has the lower and upper bound of about 49.03 and 51.00 MPa respectively where we can expect the population mean to lie in with 95% confidence. This means 95% of samples are going to contain the population mean. In order to say 99% of samples will contain the population mean, the interval has to be wider.

## LE7-3. Estimate a population percentage (2 pts)

*Adapted from Chapter 21, Review Exercise 2 (printed p. 391; PDF p. 17).*

In the chapter's historical survey, a simple random sample of **500 households** from a town of **25,000 households** included **239 with internet access**.

Use Python to calculate the estimated proportion and its estimated SE:

$$\hat p=\frac{x}{n},\qquad
\widehat{\operatorname{SE}}(\hat p)=\sqrt{\frac{\hat p(1-\hat p)}{n}}.$$

For this large-sample proportion, use the normal approximation with `z_star = stats.norm.ppf(0.975)`:

$$\hat p\pm z^*\widehat{\operatorname{SE}}(\hat p).$$

Ignore the finite-population correction; the sample includes only 2% of the town's households.

Display the estimated **percentage**, its SE in **percentage points**, and the approximate **95% confidence interval as percentages**. Interpret the interval for all households in the town in one sentence.

**Answer:**


In [11]:
total_households = 25_000
sampled_households = 500
with_internet = 239

# Calculate p_hat and its estimated SE using np.sqrt().
p_hat = with_internet / sampled_households
est_SE = np.sqrt((p_hat*(1-p_hat))/sampled_households)
# Find z_star and calculate the lower and upper confidence limits.
z_star = stats.norm.ppf(0.975)
lower_limit = p_hat - z_star*est_SE
upper_limit = p_hat + z_star*est_SE

# Display the estimate, SE, and interval in percentage units.
#split up into nparray and series so I only had to multiply by 100 once...
conf_hs_nparray = np.array([
    p_hat,
    est_SE,
    lower_limit,
    upper_limit
])
confidence_households = pd.Series(conf_hs_nparray*100,
    index = ["estimate (%)","standard error (% (units))","lower confidence limit (%)","upper confidence limit (%)"]
)
display(confidence_households)

estimate (%)                  47.8000
standard error (% (units))     2.2339
lower confidence limit (%)    43.4216
upper confidence limit (%)    52.1784
dtype: float64

In 95% of samples with 500 households, the confidence interval will contain the population mean for households with internet access. We can expect the population mean (for all 25000 houses) to lie within the interval of about 43.4 to 52.2 percentage points with 97.5 % confidence.

## Session records

Paste the block from **"make my record"** after each working session. Keep all session records in order.


<!-- Paste your session record blocks below, one after another. -->

*(session records go here)*
{
  "schema_version": "engr-tutor-export-2",
  "assignment": "WA7",
  "export_n": 1,
  "session_date": "2026-10-20",
  "messages": [
    {"i": 1, "who": "student", "code": true, "text": "can you check question 1: inspection = pd.DataFrame({\n    \"method\": [\"A\", \"A\", \"B\", \"B\"],\n    \"part_type\": [\"Simple\", \"Complex\", \"Simple\", \"Complex\"],\n    \"inspected\": [20, 80, 80, 20],\n    \"passed\": [18, 48, 64, 10],\n})\n# Calculate and display pass rates within each part type.\ninspection[\"pass_rate\"] = inspection[\"passed\"]/inspection[\"inspected\"]\ndisplay(inspection[[\"method\",\"part_type\",\"pass_rate\"]])\n\n# Group by method, sum inspected/passed counts, and calculate overall rates.\n#still has the pass rates column which is now wrong because the rates should not be summed\ngrouped_inspection = inspection.groupby(\"method\").sum()\n\n#overwrite the pass_rate column with pass rates by method group\ngrouped_inspection[\"pass_rate\"] = grouped_inspection[\"passed\"]/grouped_inspection[\"inspected\"]\ndisplay(grouped_inspection[\"pass_rate\"])\n....and then my written response is \"When split up into simple and complex parts in the original data, you can see that the simple parts have a higher pass rate than complex parts for both methods. Also, the pass rate by type is higher for each in method A than method B. When part types are combined, however, the trend reverses and it looks like method B has a higher overall pass rate. The fact that the method b group contained more simple parts that tended to pass more contributed to this."},
    {"i": 2, "who": "tutor", "text": "Your code does what the prompt asks. The within-type rates are passed divided by inspected, and the overall rates come from summed counts grouped by method. Overwriting the stale summed-rate column with a fresh division was the right fix, and the counts in that grouped table were never affected. Your comments show you caught that yourself, which is the kind of check I'd hope for.\n\n**Your comparison statements** match the numbers: the within-type verdicts for both part types and the overall verdict all agree with the data. Your composition point is also on the right track, since you name which method had more of the easier part type.\n\n**Your written answer** has two gaps against what the prompt asks:\n\n1. **Length.** The prompt asks for two sentences explaining the reversal and the causal limit. Your paragraph is longer because the comparison statements are mixed in with the explanation, so you may want to separate the statements from the two-sentence explanation.\n2. **The causal half is missing.** Nothing yet says why these observations can't establish a causal effect of method. The prompt stresses that methods were not randomly assigned. What does that tell you about how each method came to be paired with the parts it was given, and what else could differ between those parts besides the method?\n\nYour composition sentence also covers only method B's mix. Do you want to say anything about how method A's inspections divide between the two part types, or does the sentence do its job without it? That's your call.\n\nIf the weighted-average idea behind the reversal is still fuzzy, I'm happy to explain it with an invented example."},
    {"i": 3, "who": "student", "code": true, "text": "can you check question two: strength = np.array([\n    48.2, 51.4, 49.7, 52.1, 46.8, 50.5, 53.0, 47.6,\n    49.1, 51.8, 48.9, 50.7, 47.9, 52.5, 49.8, 50.2,\n])\n\n# Calculate and display n, sample_mean, sample_sd, and estimated_se.\nstrength_stats = pd.Series([\n    strength.size,\n    strength.mean(),\n    strength.std(ddof=1),\n    stats.sem(strength, ddof = 1)\n],\n    index = [\"Count\",\"Mean\",\"Sample s\",\"Standard Error\"]\n)\ndisplay(strength_stats)\nn = strength_stats[\"Count\"]    ...and my written response is \"Standard deviation, s, measures the spread of one sample around the population mean. Meanwhile the standard error, SE, is the standard deviation of the sample mean across multiple samples."},
    {"i": 4, "who": "tutor", "text": "Your code does what part A asks. You calculate `n` with `.size`, take `s` with an explicit `ddof=1`, take the SE from `stats.sem` with `ddof=1`, and nothing is rounded. The labeled `pd.Series` is a fine way to display all four values. Because a Series holds one data type, the count is stored as a float and displays with decimals. That's expected, and `n - 1` still works as a `df` in part B. The cell's comment names `sample_mean`, `sample_sd`, and `estimated_se` as well, so you may want to define those from your Series for part B.\n\n**Your written answer** is close on the SE half. For the SD half, look at the formula for `s`: it measures deviations from which mean? Check that against your phrase \"around the population mean\". Also, \"spread of one sample\" is a little loose. What exactly is `s` the spread of?\n\nThe SE half is on the right track. Think about whether your sentence says what the SE is *for* here: you computed it from a single sample, so what does the number tell you about the sample mean? The prompt also asks for one sentence, and yours currently has two.\n\nIf you'd like the SD versus SE distinction explained with an invented example, just ask."},
    {"i": 5, "who": "student", "code": true, "text": "can you check part b :# Calculate the t critical value and margin of error.\nalpha = 1-0.95\nt_star = stats.t.ppf(1-alpha/2, df = n-1)\nprint(t_star)\nlower_endpt = strength_stats[\"Mean\"] - t_star*(strength_stats[\"Sample s\"]/np.sqrt(n))\nupper_endpt = strength_stats[\"Mean\"] + t_star*(strength_stats[\"Sample s\"]/np.sqrt(n))\nME = t_star*strength_stats[\"Standard Error\"]\n# Display df, t_star, and the lower/upper confidence limits in MPa.\nconfidence_stats = pd.Series([\n    (n-1),\n    t_star,\n    lower_endpt,\n    upper_endpt\n],\n    index = [\"degrees of freedom\",\"t *\",\"lower confidence limit (MPa)\",\"upper confidence limit (MPa)\"]\n)\ndisplay(confidence_stats)    ...and my written response is \"99% interval would be wider because then using ppf the t_star value would go closer to the edges."},
    {"i": 6, "who": "tutor", "text": "Your code does what part B asks. `t_star` comes from `stats.t.ppf` with `df=n-1`, and `1 - alpha/2` works out to 0.975 for a 95% interval. The endpoints use the mean, the multiplier, and s/√n, which is the SE. Nothing is rounded before use, the display includes df, t*, and both limits, and the labels carry the MPa units. Defining `alpha` makes the link between confidence level and quantile visible, which is a nice touch.\n\nTwo small things, neither a problem:\n- You define `ME` from the SE but build the endpoints from `s/np.sqrt(n)` separately. You get the same numbers, but you could use `ME` in the endpoints so the margin is calculated once.\n- The prompt prints the quantile call as `stats.t.ppf(0.975, df=n-1)`. Yours is equivalent, so I'm just flagging it.\n\n**Your written answer** covers only one of the three things the prompt asks for.\n\n1. **The interpretation of the interval is missing.** Which quantity is the interval about, and what is it meant to capture? Your sentence should rest on your own endpoints.\n2. **The question about individual strengths is unanswered.** It asks whether the interval describes where 95% of individual strengths fall. To decide, compare the interval's width with how spread out the sixteen strengths are, and think about what the SE measures compared with s.\n3. **The 99% question:** your verdict is right. The reasoning is vague, though. What does \"closer to the edges\" do to the size of t*, and what does that do to the margin of error? Try saying that chain in your own words.\n\nIf the idea of what the 95% refers to is fuzzy, I'm happy to explain it on an invented example."},
    {"i": 7, "who": "student", "text": "is it that we expect 99 or 95% of all samples to contain pop mean?"},
    {"i": 8, "who": "tutor", "text": "Yes, that's the right idea, and it's the core of what a confidence level means.\n\nThe 95% describes the **procedure**, not any one interval. If you repeated the whole process many times (draw a fresh random sample, compute x̄ and s, build x̄ ± t*·s/√n), about 95% of the intervals you built would contain the true population mean μ, and about 5% would miss it. For a 99% interval, the same logic applies with 99% of intervals capturing μ.\n\nThe subtle part is what it does *not* say. μ is a fixed number, not a moving target. Once you have computed one specific interval from your one sample, that interval either contains μ or it doesn't. So the 95% is not \"the probability μ lies in this particular interval.\" It is the long-run capture rate of the method that produced the interval.\n\nA quick invented example: suppose a pollster surveys 1,000 randomly chosen voters in each of 200 different cities, building a 95% interval for the city's average commute time every time. If the true averages were somehow known, you'd expect roughly 190 of the 200 intervals to contain their city's true average, and about 10 to miss. Any single city's interval is one draw from that process.\n\nThis also connects to your 99% answer. A higher confidence level means the method must capture μ more often, and the way it does that is by using a larger multiplier, which widens the interval. The tradeoff is that you buy more reliability with less precision.\n\nDoes the distinction between \"this interval\" and \"the procedure that makes intervals\" feel clear? If you want, ask me to go further on any piece of it."},
    {"i": 9, "who": "student", "text": "how does the specific interval I found relate to the percent of samples"},
    {"i": 10, "who": "tutor", "text": "Your specific interval is one result from the procedure, and the 95% describes the procedure, not that one result.\n\nThink of it in two stages. **Before** you collect the sample, the endpoints are random: a different sample would give a different x̄ and s, and so different endpoints. At that stage it is fair to say there is a 95% chance the interval you are about to build will capture μ. **After** you have computed it, the endpoints are fixed numbers, and μ is also a fixed (unknown) number. Either your interval contains μ or it doesn't, and you can't tell which from the sample alone. The 95% no longer attaches to this interval as a probability.\n\nSo your interval relates to the percentage like this: it is one member of a hypothetical large collection of intervals built the same way, and roughly 95% of that collection captures μ. Since yours was built by the same method, you have 95% **confidence** in it, meaning confidence in the method that produced it. That is why confidence wording is the safe way to talk about it, rather than saying μ has a 95% chance of lying between your two numbers.\n\nA small invented example: a bakery weighs 25 randomly chosen loaves each day for 100 days and builds a 95% interval for the true mean loaf weight each day. Every day's interval has different endpoints, and about 95 of the 100 would contain the true mean. If you only saw day 37's interval, you wouldn't know whether it was one of the 95 or one of the 5. You would just know it came from a method that works 95% of the time.\n\nDoes the before-and-after distinction make sense? Once it does, you can decide how you want to phrase your own interpretation of the interval for the population mean."},
    {"i": 11, "who": "student", "code": true, "text": "can you check question 3: total_households = 25_000\nsampled_households = 500\nwith_internet = 239\n\n# Calculate p_hat and its estimated SE using np.sqrt().\np_hat = with_internet / sampled_households\nest_SE = np.sqrt((p_hat*(1-p_hat))/sampled_households)\n# Find z_star and calculate the lower and upper confidence limits.\nz_star = stats.norm.ppf(0.975)\nlower_limit = p_hat - z_star*est_SE\nupper_limit = p_hat + z_star*est_SE\n\n# Display the estimate, SE, and interval in percentage units.\n#split up into nparray and series so I only had to multiply by 100 once...\nconf_hs_nparray = np.array([\n    p_hat,\n    est_SE,\n    lower_limit,\n    upper_limit\n])\nconfidence_households = pd.Series(conf_hs_nparray*100,\n    index = [\"estimate (%)\",\"standard error (%)\",\"lower confidence limit (%)\",\"upper confidence limit (%)\"]\n)\ndisplay(confidence_households)   ... and written response \"In 95% of samples with 500 households, the confidence interval will contain the population mean for households with internet access."},
    {"i": 12, "who": "tutor", "text": "Your code does what LE7-3 asks. `p_hat` uses the sampled count over the sample size, the SE has `sampled_households` under the square root with `np.sqrt()`, `z_star` comes from `stats.norm.ppf(0.975)`, and nothing is rounded before the limits are built. Scaling the whole array by 100 once is a tidy way to get everything into percentage units, and the finite-population correction is correctly left out. One small labeling point: the SE is in percentage points, and your label says \"(%)\". The prompt asks for percentage points, so you may want the label to say that.\n\n**Your written answer** is accurate about the method, but it does not yet do what the prompt asks, which is to interpret the interval for all households in the town. Two things to look at:\n\n1. **Which interval does the sentence describe?** Yours talks about 95% of samples, which is the procedure. The prompt wants a statement about your interval for the town. Think back to the before-and-after distinction we just went through: which wording ties the 95% to the method, and what would you then say about the town's households?\n2. **What is the parameter here?** \"Population mean\" is not quite the right quantity for this problem, since you estimated a percentage. What is the population percentage a percentage of, and who is in that population?\n\nAlso, your interval's endpoints are not in the sentence. The sentence should rest on your own two limits, with units.\n\nIf you'd like, I can explain on an invented example how an interpretation names its population and its units."}
  ],
  "events": [
    {"i": 1, "part": "7-1", "type": "code"},
    {"i": 3, "part": "7-2A", "type": "code"},
    {"i": 5, "part": "7-2B", "type": "code"},
    {"i": 7, "part": "7-2B", "type": "explain"},
    {"i": 9, "part": "7-2B", "type": "explain"},
    {"i": 11, "part": "7-3", "type": "code"}
  ],
  "parts": [
    {"part": "7-1", "exchanges": 1, "max_tier": 1, "passed": false},
    {"part": "7-2A", "exchanges": 1, "max_tier": 1, "passed": false},
    {"part": "7-2B", "exchanges": 3, "max_tier": 1, "passed": false},
    {"part": "7-3", "exchanges": 1, "max_tier": 1, "passed": false}
  ],
  "tutor_assessment": [
    "The student stalled longest on 7-2B, specifically on what a 95% confidence level refers to (the procedure versus this particular interval) and how that shapes the interpretation. Their code was correct on every part; the open items were written answers: the causal half in 7-1, the s versus SE wording in 7-2A, the interpretation and individual-strengths answer in 7-2B, and population and units in 7-3.",
    "No unstuck or release-valve use occurred, and nothing looked like a tutoring failure. The two concept questions in 7-2B were answered directly and appeared to move the student toward the interpretation wording."
  ]
}

## Before you submit

1. **Restart Kernel and Run All Cells**; confirm there are no errors.
2. Check that calculations, tables, and written answers are visible.
3. Include your session record block(s).
4. Save and submit the completed `.ipynb` file in Canvas.
